<a href="https://colab.research.google.com/github/ennaguiyoussef/otto-recommender-system/blob/main/notebooks/run_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OTTO – Multi-Objective Recommender System (Colab Runner)

2-stage recommender pipeline (**Polars** + **3-Matrix Co-visitation Candidate Generation** + **LightGBM & CUDA GPU XGBoost Reranker Ensemble**).

**Recommended Colab Runtime:** `Runtime -> Change runtime type -> L4 GPU (or A100 / T4 GPU) + High-RAM ON`.

**Storage Architecture:** All datasets, intermediate Parquet artifacts, trained models, and submission files are persisted directly on Google Drive under `/content/drive/MyDrive/OTTO/`.

## 1. Mount Google Drive

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Clone or Update Repository

In [3]:
import os

# For a private repo, store a GitHub Personal Access Token in Colab Secrets as 'GITHUB_TOKEN'
try:
    from google.colab import userdata
    gh_token = userdata.get("GITHUB_TOKEN")
except Exception:
    gh_token = None

REPO_URL = (
    f"https://{gh_token}@github.com/ennaguiyoussef/otto-recommender-system.git"
    if gh_token
    else "https://github.com/ennaguiyoussef/otto-recommender-system.git"
)
REPO_DIR = "/content/otto-recommender-system"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print(f"Repository already exists at {REPO_DIR}. Pulling latest changes...")
    %cd {REPO_DIR}
    !git pull
else:
    print(f"Cloning repository into {REPO_DIR}...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

Cloning repository into /content/otto-recommender-system...
Cloning into '/content/otto-recommender-system'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 125 (delta 54), reused 111 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 71.23 KiB | 364.00 KiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/otto-recommender-system


## 3. Install Dependencies

In [4]:
!pip install -q -U kaggle
!pip install -q -r requirements.txt

## 4. Configure Kaggle API Token & Download Parquet Dataset to Google Drive

Reads your Kaggle API token (`KGAT_...`) from `/content/drive/MyDrive/OTTO/kaggle_token.txt`, configures `~/.kaggle/access_token` + `KAGGLE_API_TOKEN`, and downloads `radek1/otto-full-optimized-memory-footprint` directly into `/content/drive/MyDrive/OTTO/data/raw/` if not already present.

In [5]:
import glob
import os

BASE_DRIVE_DIR = "/content/drive/MyDrive/OTTO"
KAGGLE_TOKEN_SRC = os.path.join(BASE_DRIVE_DIR, "kaggle_token.txt")
RAW_DATA_DIR = os.path.join(BASE_DRIVE_DIR, "data", "raw")
DATASET_SLUG = "radek1/otto-full-optimized-memory-footprint"

os.makedirs(RAW_DATA_DIR, exist_ok=True)

# Configure Kaggle API Token (KGAT_...)
if not os.path.exists(KAGGLE_TOKEN_SRC):
    raise FileNotFoundError(
        f"Missing Kaggle API token file at {KAGGLE_TOKEN_SRC}."
    )

with open(KAGGLE_TOKEN_SRC, "r", encoding="utf-8") as f:
    kaggle_token = f.read().strip()

os.environ["KAGGLE_API_TOKEN"] = kaggle_token
kaggle_cfg_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_cfg_dir, exist_ok=True)
access_token_dst = os.path.join(kaggle_cfg_dir, "access_token")
with open(access_token_dst, "w", encoding="utf-8") as f:
    f.write(kaggle_token)
os.chmod(access_token_dst, 0o600)
print("Kaggle API token configured successfully.")

# Download optimized Parquet dataset if not already present on Drive
existing_parquets = glob.glob(os.path.join(RAW_DATA_DIR, "**", "*.parquet"), recursive=True)
if existing_parquets:
    print(f"Found {len(existing_parquets)} Parquet file(s) in {RAW_DATA_DIR}. Skipping download.")
else:
    print(f"Downloading {DATASET_SLUG} into {RAW_DATA_DIR}...")
    !kaggle datasets download -d {DATASET_SLUG} -p "{RAW_DATA_DIR}" --unzip
    print("Download and extraction complete!")

Kaggle API token configured successfully.
Found 2 Parquet file(s) in /content/drive/MyDrive/OTTO/data/raw. Skipping download.


## 5. Run End-to-End Pipeline (`sample` Mode)

In [6]:
!python main.py --mode sample --stage train

2026-11-25 19:35:05 | INFO     | otto.main | Starting OTTO pipeline | mode='sample' | stage='train' | force=False | config='configs/config.yaml'
2026-11-25 19:35:05 | INFO     | src.utils.config_loader | Initialized storage directories under: /content/drive/MyDrive/OTTO
2026-11-25 19:35:05 | INFO     | otto.main | === Running stage: TRAIN ===
2026-11-25 19:35:07 | INFO     | src.models.train | [checkpoint] Upgrading 2-way metadata to 3-way ensemble using existing model checkpoints...
2026-11-25 19:35:07 | INFO     | src.models.train | [train] Detected NVIDIA GPU: GPU 0: NVIDIA L4 (UUID: GPU-1e7c2051-dbfb-ac4d-1f8f-44752f4ff8a0)
2026-11-25 19:35:07 | INFO     | src.models.train | [train] XGBoost Ranker active on device='cuda'
2026-11-25 19:35:07 | INFO     | src.models.train | [train] Loading validation candidate features from '/content/drive/MyDrive/OTTO/artifacts/features/sample/features.parquet'
2026-11-25 19:35:29 | INFO     | src.models.train | [train] Loaded 8669737 rows across 90

In [ ]:
!python main.py --mode sample --stage candidates,features,train,predict

2026-11-25 18:42:27 | INFO     | otto.main | Starting OTTO pipeline | mode='sample' | stage='candidates,features,train,predict' | config='configs/config.yaml'
2026-11-25 18:42:27 | INFO     | src.utils.config_loader | Initialized storage directories under: /content/drive/MyDrive/OTTO
2026-11-25 18:42:27 | INFO     | otto.main | === Running stage: CANDIDATES ===
2026-11-25 18:44:19 | INFO     | src.candidates.covisitation | [candidates] Streaming 'train.parquet' (171266098 rows) into 16 disk buckets...
2026-11-25 18:44:56 | INFO     | src.candidates.covisitation | [candidates] Streaming 'history.parquet' (384620 rows) into 16 disk buckets...
2026-11-25 18:46:22 | INFO     | src.candidates.covisitation | [candidates] Processed session bucket 4/16
2026-11-25 18:47:46 | INFO     | src.candidates.covisitation | [candidates] Processed session bucket 8/16
2026-11-25 18:49:11 | INFO     | src.candidates.covisitation | [candidates] Processed session bucket 12/16
2026-11-25 18:50:34 | INFO     |

In [ ]:
!python main.py --mode sample --stage all

## 6. Run Full Pipeline (ull Mode — Auto-Resume Checkpoints Enabled)

All stages, buckets, feature shards, models, and prediction batches checkpoint atomically on Google Drive.
**If Colab disconnects or GPU stops mid-run:** simply reconnect (GPU or CPU) and re-run !python main.py --mode full --stage all — it automatically resumes from the exact unfinished step!

In [7]:
# Uncomment when ready to train and generate submissions on 100% of the data:
!python main.py --mode full --stage all

2026-11-25 19:39:56 | INFO     | otto.main | Starting OTTO pipeline | mode='full' | stage='all' | force=False | config='configs/config.yaml'
2026-11-25 19:39:56 | INFO     | src.utils.config_loader | Initialized storage directories under: /content/drive/MyDrive/OTTO
2026-11-25 19:39:56 | INFO     | otto.main | === Running stage: SAMPLE ===
2026-11-25 19:39:56 | INFO     | src.data.make_sample | [sample] Mode is 'full' (sample_frac=1.0); skipping subsample generation.
2026-11-25 19:39:56 | INFO     | otto.main | === Finished stage: SAMPLE (0.00s) ===
2026-11-25 19:39:56 | INFO     | otto.main | === Running stage: SPLIT ===
2026-11-25 19:39:56 | INFO     | src.data.split | [split] Splitting sessions (mode=full, val_days=7, query_frac=1.00, corpus_frac=1.00) from '/content/drive/MyDrive/OTTO/data/raw' -> '/content/drive/MyDrive/OTTO/data/splits/full'
2026-11-25 19:40:01 | INFO     | src.data.split | [split] Timeline [1659304800 .. 1661723999] (scale=1s) -> cutoff_ts=1661119199
2026-11-25 

### 7. Soumettre sur Kaggle

In [8]:
!kaggle competitions submit -c otto-recommender-system \
  -f /content/drive/MyDrive/OTTO/submissions/submission_full.csv \
  -m "2-stage: Polars 3-matrix covis + LightGBM/XGBoost ensemble (local R@20=0.5809)"

100% 783M/783M [00:38<00:00, 21.3MB/s]
Submission ref: 58909869
99 submissions remaining today.
Successfully submitted to OTTO – Multi-Objective Recommender System

### 8. Score

In [11]:
!kaggle competitions submissions -c otto-recommender-system

     ref  fileName             date                        description                                                                     status                     publicScore  privateScore  
--------  -------------------  --------------------------  ------------------------------------------------------------------------------  -------------------------  -----------  ------------  
58909869  submission_full.csv  2026-11-25 21:05:06.143000  2-stage: Polars 3-matrix covis + LightGBM/XGBoost ensemble (local R@20=0.5809)  SubmissionStatus.COMPLETE  0.59578      0.59542       
